# C11-neural-training — Practice p09 — Solution


**Type:** constrained coding · **Difficulty:** core · **Concepts:** batch-normalization


The current batch is normalized with its biased variance. The persistent
running-variance update uses the matching \(N-1\) estimate. Gamma and beta are
trainable parameters; running statistics are module-updated buffers.


In [ ]:
import numpy as np

def batchnorm_forward_audit(X, gamma, beta, running_mean, running_var, eps, momentum):
    arrays = [np.asarray(a, dtype=np.float64) for a in (X, gamma, beta, running_mean, running_var)]
    Xv, g, b, rm, rv = arrays
    mean = Xv.mean(axis=0)
    centered = Xv - mean
    biased = np.mean(centered**2, axis=0)
    unbiased = np.sum(centered**2, axis=0) / (Xv.shape[0] - 1)
    train = g * centered / np.sqrt(biased + eps) + b
    rm_new = (1.0-momentum)*rm + momentum*mean
    rv_new = (1.0-momentum)*rv + momentum*unbiased
    eval_output = g * (Xv-rm_new) / np.sqrt(rv_new+eps) + b
    return {"train_output": train, "eval_output": eval_output, "batch_mean": mean,
            "batch_var_biased": biased, "running_mean_new": rm_new, "running_var_new": rv_new,
            "parameter_names": ("gamma", "beta"), "buffer_names": ("running_mean", "running_var")}

X_p09 = np.array([[1.0, 4.0, 7.0], [3.0, 4.0, 9.0], [5.0, 4.0, 11.0], [7.0, 4.0, 13.0]])
gamma_p09 = np.array([1.2, -0.7, 0.5])
beta_p09 = np.array([0.1, 0.2, -0.3])
running_mean_p09 = np.array([0.5, -0.5, 1.0])
running_var_p09 = np.array([1.5, 0.75, 2.0])
result_p09 = batchnorm_forward_audit(X_p09, gamma_p09, beta_p09, running_mean_p09, running_var_p09, eps=1e-5, momentum=0.1)


### Answer check


In [ ]:
inputs_ref_p09 = (
    np.array([[1.0, 4.0, 7.0], [3.0, 4.0, 9.0], [5.0, 4.0, 11.0], [7.0, 4.0, 13.0]]),
    np.array([1.2, -0.7, 0.5]),
    np.array([0.1, 0.2, -0.3]),
    np.array([0.5, -0.5, 1.0]),
    np.array([1.5, 0.75, 2.0]),
)
mean_ref_p09 = inputs_ref_p09[0].sum(axis=0) / 4
centered_ref_p09 = inputs_ref_p09[0] - mean_ref_p09
var_biased_ref_p09 = (centered_ref_p09**2).sum(axis=0) / 4
var_unbiased_ref_p09 = (centered_ref_p09**2).sum(axis=0) / 3
train_ref_p09 = inputs_ref_p09[1] * centered_ref_p09 / np.sqrt(var_biased_ref_p09 + 1e-5) + inputs_ref_p09[2]
running_mean_ref_p09 = 0.9 * inputs_ref_p09[3] + 0.1 * mean_ref_p09
running_var_ref_p09 = 0.9 * inputs_ref_p09[4] + 0.1 * var_unbiased_ref_p09
eval_ref_p09 = inputs_ref_p09[1] * (inputs_ref_p09[0] - running_mean_ref_p09) / np.sqrt(running_var_ref_p09 + 1e-5) + inputs_ref_p09[2]

assert np.allclose(result_p09["batch_mean"], mean_ref_p09, atol=1e-10, rtol=1e-9)
assert np.allclose(result_p09["batch_var_biased"], var_biased_ref_p09, atol=1e-10, rtol=1e-9)
assert result_p09["batch_var_biased"][1] == 0.0
assert np.allclose(result_p09["train_output"], train_ref_p09, atol=1e-10, rtol=1e-9)
assert np.allclose(result_p09["running_mean_new"], running_mean_ref_p09, atol=1e-10, rtol=1e-9)
assert np.allclose(result_p09["running_var_new"], running_var_ref_p09, atol=1e-10, rtol=1e-9)
assert np.allclose(result_p09["eval_output"], eval_ref_p09, atol=1e-10, rtol=1e-9)
assert result_p09["parameter_names"] == ("gamma", "beta")
assert result_p09["buffer_names"] == ("running_mean", "running_var")
for actual_p09, expected_input_p09 in zip(
    (X_p09, gamma_p09, beta_p09, running_mean_p09, running_var_p09), inputs_ref_p09
):
    assert np.array_equal(actual_p09, expected_input_p09)
